# Tugas 3: Modelling Klasifikasi Berita Menggunakan Orange Data Mining

Pada tugas ini, dilakukan pemodelan menggunakan aplikasi **Orange Data Mining** untuk mengklasifikasikan teks berita ke dalam kategori *Sport* dan *Finance*.

Alur pemodelan dibagi menjadi dua skenario eksperimen:
1. **Alur Node Pertama (Alur Atas)**: Pemodelan data TF-IDF dengan reduksi dimensi menggunakan teknik proyeksi matematis **Principal Component Analysis (PCA)**.
2. **Alur Node Kedua (Alur Bawah)**: Pemodelan data TF-IDF dengan reduksi seleksi leksikal berbasis frekuensi kata teratas (**500 fitur leksikal**).

---

## 1. Alur Pembuatan Node Pertama (Reduksi Dimensi PCA)

Berikut adalah susunan node-node pada kanvas Orange Data Mining untuk alur pemodelan data pertama (alur atas):

![Alur Node Orange](images/orange_workflow.png)

**Penjelasan Langkah Pembuatan Node:**
1. **CSV File Import**: Mengimpor file dataset TF-IDF berita (`tfidf_berita_train.csv` / `tfidf_berita.csv`).
2. **Select Columns (1)**: Menentukan peran masing-masing atribut, memisahkan fitur prediktor dan variabel target.
3. **PCA**: Melakukan reduksi ruang vektor data teks yang berdimensi ribuan menjadi ruang komponen utama yang lebih ringkas.
4. **Model Klasifikasi (Naive Bayes & kNN)**: Kedua model dihubungkan dari node *PCA* untuk menerima data hasil reduksi komponen utama.
5. **Test and Score (1)**: Menerima saluran *Learner* dari model serta saluran *Data* dari PCA untuk mengevaluasi performa klasifikasi.

## 2. Pengaturan Kolom Target pada Select Columns

Tahap penting pada widget **Select Columns** adalah memisahkan kolom fitur dan kolom target sebelum diproses oleh PCA dan model:

![Pengaturan Select Columns](images/orange_select_columns.png)

**Penjelasan Pengaturan Kolom:**
- **Target Variable**: Kolom `label` ditempatkan pada kotak **Target** sebagai kelas yang akan diprediksi (*sport* dan *finance*).
- **Features**: Seluruh kolom kata ditempatkan pada kotak **Features** sebagai atribut prediktor.
- **Metas**: Kolom indeks dokumen (`Unnamed: 0`) ditempatkan pada kotak **Metas** sebagai pelengkap informasi tanpa memengaruhi komputasi model.

## 3. Pengaturan Parameter PCA (Explained Variance 80%)

Pada widget **PCA**, dilakukan penyesuaian parameter agar tidak hanya mempertahankan 2 komponen bawaan (yang hanya memuat 5% variansi informasi), melainkan mempertahankan mayoritas variansi data teks:

![Pengaturan PCA](images/orange_pca_settings.png)

**Penjelasan Konfigurasi PCA:**
- **Explained Variance = 80%**: Memilih ambang batas variansi kumulatif sebesar 80% untuk memastikan bahwa sebagian besar informasi teks tetap terjaga.
- **Components Terpilih Secara Otomatis = 73**: Berdasarkan grafik garis hijau (*cumulative variance*), Orange secara otomatis menentukan bahwa dibutuhkan **73 komponen utama** untuk merangkum 80% variansi informasi korpus berita.
- **Normalize variables**: Dicentang (*aktif*) agar seluruh variabel fitur distandarisasi skalanya sebelum diekstraksi komponen utamanya.
- **Efisiensi Reduksi**: Dari ribuan fitur kata awal, PCA berhasil mereduksi ruang fitur hingga ~98.8% (hanya menyisakan 73 komponen) namun tetap mempertahankan 80% esensi informasi teks.

## 4. Hasil Pengujian dan Evaluasi Node Pertama (Test and Score 1)

Pengujian performa model setelah reduksi PCA 80% (73 komponen) dilakukan menggunakan widget **Test and Score (1)**:

![Hasil Test and Score Node 1](images/orange_test_score.png)

**Parameter Evaluasi:**
- **Metode Pengujian**: **Random sampling** dengan 10 kali pengulangan (*Repeat train/test: 10*).
- **Proporsi Latih & Uji**: 66% data latih dan 34% data uji secara berimbang (*Stratified*).
- **Data Masukan**: 120 dokumen latih (`->] 120`), menghasilkan 410 evaluasi sampel uji (`[-> 410`).

**Tabel Metrik Evaluasi Node Pertama (PCA 80% / 73 Komponen):**

| Model | AUC | CA (Accuracy) | F1-Score | Precision | Recall | MCC |
|---|---|---|---|---|---|---|
| **Naive Bayes** | 1.000 | **0.993** (99.3%) | **0.993** | **0.993** | **0.993** | **0.985** |
| **kNN** | 1.000 | **0.988** (98.8%) | **0.988** | **0.988** | **0.988** | **0.976** |

**Analisis Hasil Node Pertama:**
- **Naive Bayes**: Menghasilkan akurasi yang sangat stabil di angka **99.3%** dan F1-score **0.993**, menunjukkan bahwa distribusi probabilitas kelas tetap terpisah sangat baik pada 73 komponen PCA.
- **kNN**: Menghasilkan akurasi **98.8%** dan MCC **0.976**, menunjukkan performa klasifikasi tetangga terdekat yang sangat tinggi dan realistis pada ruang proyeksi berdimensi 73.
- **AUC = 1.000**: Kedua model mempertahankan nilai *Area Under ROC Curve* maksimal, menandakan kemampuan separasi sempurna antar-kelas *Sport* dan *Finance*.

## 5. Alur Pembuatan Node Kedua (Data Reduksi Leksikal 500 Fitur)

Alur pemodelan kedua (alur bawah) diterapkan pada dataset TF-IDF yang telah direduksi secara leksikal sejak awal menjadi 500 fitur kata teratas (`tfidf_berita_500.csv`):

![Alur Node Orange Reduksi](images/orange_workflow_reduksi.png)

**Penjelasan Langkah Pembuatan Node:**
1. **CSV File Import (1)**: Mengimpor file dataset TF-IDF 500 fitur leksikal.
2. **Select Columns**: Menetapkan kolom `label` sebagai **Target**, 500 kolom kata sebagai **Features**, dan indeks dokumen sebagai **Metas**.
3. **Model Klasifikasi (Naive Bayes & kNN)**: Langsung dihubungkan dari *Select Columns* tanpa memerlukan node perantara karena dimensi data masukan sudah ringkas (500 kata).
4. **Test and Score**: Melakukan pengujian performa model dengan parameter *Random sampling* yang identik.

## 6. Hasil Pengujian dan Evaluasi Node Kedua (Test and Score 2)

Evaluasi performa klasifikasi untuk data reduksi leksikal 500 fitur disajikan pada widget **Test and Score** alur kedua:

![Hasil Test and Score Reduksi](images/orange_test_score_reduksi.png)

**Tabel Metrik Evaluasi Node Kedua (500 Fitur Leksikal):**

| Model | AUC | CA (Accuracy) | F1-Score | Precision | Recall | MCC |
|---|---|---|---|---|---|---|
| **kNN** | 1.000 | **0.998** (99.8%) | **0.998** | **0.998** | **0.998** | **0.995** |
| **Naive Bayes (1)** | 1.000 | **0.995** (99.5%) | **0.995** | **0.995** | **0.995** | **0.990** |

**Analisis Hasil Node Kedua:**
- Model **kNN** mencapai tingkat akurasi puncak sebesar **99.8%** dengan nilai korelasi Matthews (**MCC**) mencapai **0.995**.
- Model **Naive Bayes** mencapai akurasi **99.5%** dengan F1-score **0.995** dan MCC **0.990**.

## 7. Perbandingan Komprehensif: Reduksi PCA (73 Komponen) vs Reduksi Leksikal (500 Fitur)

Tabel di bawah ini merangkum perbandingan performa antara kedua pendekatan reduksi dimensi:

| Skenario Pemodelan | Pendekatan Reduksi | Jumlah Fitur | Model | AUC | CA (Accuracy) | F1-Score | Precision | Recall | MCC |
|---|---|---|---|---|---|---|---|---|---|
| **Node 1 (Alur Atas)** | PCA (Explained Var 80%) | 73 komponen | Naive Bayes | 1.000 | **0.993** | 0.993 | 0.993 | 0.993 | 0.985 |
| **Node 1 (Alur Atas)** | PCA (Explained Var 80%) | 73 komponen | kNN | 1.000 | **0.988** | 0.988 | 0.988 | 0.988 | 0.976 |
| **Node 2 (Alur Bawah)** | Leksikal (*Top Term Frequency*) | 500 kata | Naive Bayes | 1.000 | **0.995** | 0.995 | 0.995 | 0.995 | 0.990 |
| **Node 2 (Alur Bawah)** | Leksikal (*Top Term Frequency*) | 500 kata | kNN | 1.000 | **0.998** | 0.998 | 0.998 | 0.998 | 0.995 |

**Kesimpulan dan Temuan Penelitian:**
1. **Keunggulan Reduksi Dimensi**: Kedua metode reduksi (baik PCA 73 komponen maupun seleksi leksikal 500 kata) terbukti sangat efektif, mempertahankan akurasi klasifikasi di atas **98.8% - 99.8%**.
2. **Efisiensi Kompresi PCA**: Pendekatan **PCA** berhasil memadatkan korpus dari ribuan kata menjadi hanya **73 dimensi fitur kontinu** (~98.8% reduksi ruang fitur) dengan penurunan performa yang sangat minimal (Naive Bayes tetap 99.3% dan kNN di 98.8%).
3. **Karakteristik Reduksi Leksikal**: Seleksi 500 kata teratas menghasilkan akurasi yang sedikit lebih unggul (kNN 99.8%) karena mempertahankan kata-kata kunci asli (*domain-specific terms*) yang sangat khas bagi berita *Sport* dan *Finance*.
4. **Rekomendasi Model**: Untuk kebutuhan efisiensi memori ekstrem dan ruang data padat (*dense*), **Naive Bayes dengan PCA 73 komponen** sangat direkomendasikan. Sedangkan untuk akurasi klasifikasi tertinggi, **kNN dengan data TF-IDF 500 kata leksikal** merupakan konfigurasi terbaik.